In [1]:
import torch
torch.hub.set_dir('/kaggle/working/torch_hub')

In [2]:
"""
Segmentation Training Script
Converted from train_mask.ipynb
Trains a segmentation head on top of DINOv2 backbone
"""

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
import cv2
import os
import torchvision
from tqdm import tqdm
from torchvision.transforms import InterpolationMode

# Set matplotlib to non-interactive backend
plt.switch_backend('Agg')


In [3]:

# ============================================================================
# Utility Functions
# ============================================================================

def save_image(img, filename):
    """Save an image tensor to file after denormalizing."""
    img = np.array(img)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = np.moveaxis(img, 0, -1)
    img = (img * std + mean) * 255
    cv2.imwrite(filename, img[:, :, ::-1])


# ============================================================================
# Mask Conversion
# ============================================================================

# Mapping from raw pixel values to new class IDs
value_map = {
    0: 0,        # background
    100: 1,      # Trees
    200: 2,      # Lush Bushes
    300: 3,      # Dry Grass
    500: 4,      # Dry Bushes
    550: 5,      # Ground Clutter
    700: 6,      # Logs
    800: 7,      # Rocks
    7100: 8,     # Landscape
    10000: 9     # Sky
}
n_classes = len(value_map)


def convert_mask(mask):
    """Convert raw mask values to class IDs."""
    arr = np.array(mask)
    new_arr = np.zeros_like(arr, dtype=np.uint8)
    for raw_value, new_value in value_map.items():
        new_arr[arr == raw_value] = new_value
    return Image.fromarray(new_arr)


In [4]:

# ============================================================================
# Dataset
# ============================================================================

class MaskDataset(Dataset):
    def __init__(self, data_dir, transform=None, mask_transform=None):
        self.image_dir = os.path.join(data_dir, 'Color_Images')
        self.masks_dir = os.path.join(data_dir, 'Segmentation')
        self.transform = transform
        self.mask_transform = mask_transform
        self.data_ids = os.listdir(self.image_dir)

    def __len__(self):
        return len(self.data_ids)

    def __getitem__(self, idx):
        data_id = self.data_ids[idx]
        img_path = os.path.join(self.image_dir, data_id)
        # Both color images and masks are .png files with same name
        mask_path = os.path.join(self.masks_dir, data_id)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)
        mask = convert_mask(mask)

        if self.transform:
             image = self.transform(image)
        if self.mask_transform:
             mask = self.mask_transform(mask)
        mask = torch.from_numpy(np.array(mask)).long()

        return image, mask


In [5]:

# ============================================================================
# Model: Segmentation Head (ConvNeXt-style)
# ============================================================================

# class SegmentationHeadConvNeXt(nn.Module):
#     def __init__(self, in_channels, out_channels, tokenW, tokenH):
#         super().__init__()
#         self.H, self.W = tokenH, tokenW

#         self.stem = nn.Sequential(
#             nn.Conv2d(in_channels, 128, kernel_size=7, padding=3),
#             nn.GELU()
#         )

#         self.block = nn.Sequential(
#             nn.Conv2d(128, 128, kernel_size=7, padding=3, groups=128),
#             nn.GELU(),
#             nn.Conv2d(128, 128, kernel_size=1),
#             nn.GELU(),
#         )

#         self.classifier = nn.Conv2d(128, out_channels, 1)

#     def forward(self, x):
#         B, N, C = x.shape
#         x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)
#         x = self.stem(x)
#         x = self.block(x)
#         return self.classifier(x)

# class FPNSegmentationHead(nn.Module):
#     def __init__(self, in_channels, out_channels, tokenW, tokenH):
#         super().__init__()
#         self.H, self.W = tokenH, tokenW
        
#         # Project DINOv2 features
#         self.proj = nn.Conv2d(in_channels, 256, kernel_size=1)
        
#         # FPN lateral connections at different scales
#         self.layer1 = nn.Sequential(
#             nn.Conv2d(256, 256, 3, padding=1),
#             nn.BatchNorm2d(256),
#             nn.GELU()
#         )
#         self.layer2 = nn.Sequential(
#             nn.Conv2d(256, 128, 3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.GELU()
#         )
#         self.layer3 = nn.Sequential(
#             nn.Conv2d(128, 64, 3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.GELU()
#         )
        
#         self.classifier = nn.Conv2d(64, out_channels, 1)
#         self.dropout = nn.Dropout2d(0.1)

#     def forward(self, x):
#         B, N, C = x.shape
#         x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)
        
#         x = self.proj(x)          # [B, 256, H, W]
        
#         x1 = self.layer1(x)       # [B, 256, H, W]
        
#         # Upsample 2x
#         x2 = F.interpolate(x1, scale_factor=2, mode='bilinear', align_corners=False)
#         x2 = self.layer2(x2)      # [B, 128, 2H, 2W]
        
#         # Upsample 2x again
#         x3 = F.interpolate(x2, scale_factor=2, mode='bilinear', align_corners=False)
#         x3 = self.layer3(x3)      # [B, 64, 4H, 4W]
        
#         x3 = self.dropout(x3)
#         return self.classifier(x3)

# class FPNSegmentationHead(nn.Module):
#     def __init__(self, in_channels, out_channels, tokenW, tokenH):
#         super().__init__()
#         self.H, self.W = tokenH, tokenW

#         # Projection
#         self.proj = nn.Sequential(
#             nn.Conv2d(in_channels, 256, 1),
#             nn.BatchNorm2d(256),
#             nn.GELU()
#         )

#         # Depthwise refinement block
#         self.refine = nn.Sequential(
#             nn.Conv2d(256, 256, 3, padding=1, groups=256),
#             nn.Conv2d(256, 256, 1),
#             nn.BatchNorm2d(256),
#             nn.GELU()
#         )

#         # Channel attention
#         self.attn = nn.Sequential(
#             nn.AdaptiveAvgPool2d(1),
#             nn.Conv2d(256, 64, 1),
#             nn.GELU(),
#             nn.Conv2d(64, 256, 1),
#             nn.Sigmoid()
#         )

#         # Upsampling blocks
#         self.up1 = nn.Sequential(
#             nn.Conv2d(256, 128, 3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.GELU()
#         )

#         self.up2 = nn.Sequential(
#             nn.Conv2d(128, 64, 3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.GELU()
#         )

#         self.classifier = nn.Conv2d(64, out_channels, 1)

#     def forward(self, x):
#         B, N, C = x.shape
#         x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)

#         x = self.proj(x)
#         x = self.refine(x)

#         # attention
#         attn = self.attn(x)
#         x = x * attn

#         x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
#         x = self.up1(x)

#         x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
#         x = self.up2(x)

#         return self.classifier(x)

class ASPPFPNSegmentationHead(nn.Module):
    def __init__(self, in_channels, out_channels, tokenW, tokenH):
        super().__init__()
        self.H, self.W = tokenH, tokenW
        
        # ASPP branches
        self.aspp_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 1),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d6 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=6, dilation=6),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d12 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=12, dilation=12),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.aspp_d18 = nn.Sequential(
            nn.Conv2d(in_channels, 256, 3, padding=18, dilation=18),
            nn.BatchNorm2d(256),
            nn.GELU()
        )
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, 256, 1),
            nn.GELU()
        )
        
        # Fuse all 5 ASPP branches
        self.aspp_fuse = nn.Sequential(
            nn.Conv2d(256 * 5, 256, 1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Dropout2d(0.1)
        )
        
        # FPN progressive upsampling
        self.up1 = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU()
        )
        self.up2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU()
        )
        
        self.classifier = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        B, N, C = x.shape
        x = x.reshape(B, self.H, self.W, C).permute(0, 3, 1, 2)
        
        # ASPP - all branches in parallel
        b1 = self.aspp_1x1(x)
        b2 = self.aspp_d6(x)
        b3 = self.aspp_d12(x)
        b4 = self.aspp_d18(x)
        b5 = F.interpolate(self.gap(x), size=x.shape[2:], mode='bilinear', align_corners=False)
        
        # Fuse
        x = self.aspp_fuse(torch.cat([b1, b2, b3, b4, b5], dim=1))
        
        # FPN progressive upsampling
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = self.up1(x)
        
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = self.up2(x)
        
        return self.classifier(x)
#  FOR DECODER LAYER WE HAVE ADDED BATCH NORMALIZATION AS MANY PAPERS APPLY THIS IN THERE DECODER LAYER TO STABALIZE THE FEATURES GRADIENTS AS WELL AS RESIDUAL LAYER IS ALSO USED 

In [6]:

# ============================================================================
# Metrics
# ============================================================================

def compute_iou(pred, target, num_classes=10, ignore_index=255):
    """Compute IoU for each class and return mean IoU."""
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)

    iou_per_class = []
    for class_id in range(num_classes):
        if class_id == ignore_index:
            continue

        pred_inds = pred == class_id
        target_inds = target == class_id

        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()

        if union == 0:
            iou_per_class.append(float('nan'))
        else:
            iou_per_class.append((intersection / union).cpu().numpy())

    return np.nanmean(iou_per_class)


def compute_dice(pred, target, num_classes=10, smooth=1e-6):
    """Compute Dice coefficient (F1 Score) per class and return mean Dice Score."""
    pred = torch.argmax(pred, dim=1)
    pred, target = pred.view(-1), target.view(-1)

    dice_per_class = []
    for class_id in range(num_classes):
        pred_inds = pred == class_id
        target_inds = target == class_id

        intersection = (pred_inds & target_inds).sum().float()
        dice_score = (2. * intersection + smooth) / (pred_inds.sum().float() + target_inds.sum().float() + smooth)

        dice_per_class.append(dice_score.cpu().numpy())

    return np.mean(dice_per_class)


def compute_pixel_accuracy(pred, target):
    """Compute pixel accuracy."""
    pred_classes = torch.argmax(pred, dim=1)
    return (pred_classes == target).float().mean().cpu().numpy()


def evaluate_metrics(model, backbone, data_loader, device, num_classes=10, show_progress=True):
    """Evaluate all metrics on a dataset."""
    iou_scores = []
    dice_scores = []
    pixel_accuracies = []

    model.eval()
    loader = tqdm(data_loader, desc="Evaluating", leave=False, unit="batch") if show_progress else data_loader
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            output = backbone.forward_features(imgs)["x_norm_patchtokens"]
            logits = model(output.to(device))
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)  # ADD THIS - was missing

            labels = labels.squeeze(dim=1).long()
            labels = F.interpolate(labels.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest").squeeze(1).long() 
            iou = compute_iou(outputs, labels, num_classes=num_classes)
            dice = compute_dice(outputs, labels, num_classes=num_classes)
            pixel_acc = compute_pixel_accuracy(outputs, labels)

            iou_scores.append(iou)
            dice_scores.append(dice)
            pixel_accuracies.append(pixel_acc)

    model.train()
    return np.mean(iou_scores), np.mean(dice_scores), np.mean(pixel_accuracies)


In [7]:

# ============================================================================
# Plotting Functions
# ============================================================================

def save_training_plots(history, output_dir):
    """Save all training metric plots to files."""
    os.makedirs(output_dir, exist_ok=True)

    # Plot 1: Loss curves
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'))
    plt.close()
    print(f"Saved training curves to '{output_dir}/training_curves.png'")

    # Plot 2: IoU curves
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_iou'], label='Train IoU')
    plt.title('Train IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_iou'], label='Val IoU')
    plt.title('Validation IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'iou_curves.png'))
    plt.close()
    print(f"Saved IoU curves to '{output_dir}/iou_curves.png'")

    # Plot 3: Dice curves
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_dice'], label='Train Dice')
    plt.title('Train Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_dice'], label='Val Dice')
    plt.title('Validation Dice vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'dice_curves.png'))
    plt.close()
    print(f"Saved Dice curves to '{output_dir}/dice_curves.png'")

    # Plot 4: Combined metrics plot
    plt.figure(figsize=(12, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title('Loss vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 2)
    plt.plot(history['train_iou'], label='train')
    plt.plot(history['val_iou'], label='val')
    plt.title('IoU vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 3)
    plt.plot(history['train_dice'], label='train')
    plt.plot(history['val_dice'], label='val')
    plt.title('Dice Score vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 2, 4)
    plt.plot(history['train_pixel_acc'], label='train')
    plt.plot(history['val_pixel_acc'], label='val')
    plt.title('Pixel Accuracy vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Pixel Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'all_metrics_curves.png'))
    plt.close()
    print(f"Saved combined metrics curves to '{output_dir}/all_metrics_curves.png'")


def save_history_to_file(history, output_dir):
    """Save training history to a text file."""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, 'evaluation_metrics.txt')

    with open(filepath, 'w') as f:
        f.write("TRAINING RESULTS\n")
        f.write("=" * 50 + "\n\n")

        f.write("Final Metrics:\n")
        f.write(f"  Final Train Loss:     {history['train_loss'][-1]:.4f}\n")
        f.write(f"  Final Val Loss:       {history['val_loss'][-1]:.4f}\n")
        f.write(f"  Final Train IoU:      {history['train_iou'][-1]:.4f}\n")
        f.write(f"  Final Val IoU:        {history['val_iou'][-1]:.4f}\n")
        f.write(f"  Final Train Dice:     {history['train_dice'][-1]:.4f}\n")
        f.write(f"  Final Val Dice:       {history['val_dice'][-1]:.4f}\n")
        f.write(f"  Final Train Accuracy: {history['train_pixel_acc'][-1]:.4f}\n")
        f.write(f"  Final Val Accuracy:   {history['val_pixel_acc'][-1]:.4f}\n")
        f.write("=" * 50 + "\n\n")

        f.write("Best Results:\n")
        f.write(f"  Best Val IoU:      {max(history['val_iou']):.4f} (Epoch {np.argmax(history['val_iou']) + 1})\n")
        f.write(f"  Best Val Dice:     {max(history['val_dice']):.4f} (Epoch {np.argmax(history['val_dice']) + 1})\n")
        f.write(f"  Best Val Accuracy: {max(history['val_pixel_acc']):.4f} (Epoch {np.argmax(history['val_pixel_acc']) + 1})\n")
        f.write(f"  Lowest Val Loss:   {min(history['val_loss']):.4f} (Epoch {np.argmin(history['val_loss']) + 1})\n")
        f.write("=" * 50 + "\n\n")

        f.write("Per-Epoch History:\n")
        f.write("-" * 100 + "\n")
        headers = ['Epoch', 'Train Loss', 'Val Loss', 'Train IoU', 'Val IoU',
                   'Train Dice', 'Val Dice', 'Train Acc', 'Val Acc']
        f.write("{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}\n".format(*headers))
        f.write("-" * 100 + "\n")

        n_epochs = len(history['train_loss'])
        for i in range(n_epochs):
            f.write("{:<8} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f} {:<12.4f}\n".format(
                i + 1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['train_iou'][i],
                history['val_iou'][i],
                history['train_dice'][i],
                history['val_dice'][i],
                history['train_pixel_acc'][i],
                history['val_pixel_acc'][i]
            ))

    print(f"Saved evaluation metrics to {filepath}")


In [8]:
class CombinedLoss(nn.Module):
    def __init__(self, num_classes, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss()
        self.num_classes = num_classes

    def dice_loss(self, pred, target):
        pred = F.softmax(pred, dim=1)
        loss = 0
        for c in range(self.num_classes):
            pred_c = pred[:, c]
            target_c = (target == c).float()
            intersection = (pred_c * target_c).sum()
            loss += 1 - (2 * intersection + 1e-6) / (pred_c.sum() + target_c.sum() + 1e-6)
        return loss / self.num_classes

    def forward(self, pred, target):
        return self.alpha * self.ce(pred, target) + (1 - self.alpha) * self.dice_loss(pred, target)

In [9]:


# ============================================================================
# Main Training Function
# ============================================================================

def main():
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Hyperparameters
    batch_size = 4
    w = int(((960 / 2) // 14) * 14)
    h = int(((540 / 2) // 14) * 14)
    lr = 1e-4
    n_epochs = 10

    # Output directory (relative to script location)
    # script_dir = os.path.dirname(os.path.abspath(__file__))// IN NORMAL 
    script_dir = "/kaggle/working"
    output_dir = os.path.join(script_dir, 'train_stats')
    os.makedirs(output_dir, exist_ok=True)

    # Transforms
    transform = transforms.Compose([
        transforms.Resize((h, w)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    # ADD THIS - for training only
    transform_train = transforms.Compose([
       transforms.Resize((h, w)),  # deterministic → safe
       transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
       transforms.RandomGrayscale(p=0.1),
       transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
       transforms.ToTensor(),
       transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
    ])

    mask_transform = transforms.Compose([
           transforms.Resize((h, w), interpolation=InterpolationMode.NEAREST),
           ])

    # Dataset paths (relative to script location)
    # data_dir = os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'train')
    # val_dir = os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'val')
    data_dir = "/kaggle/input/datasets/divyanshsharma23/semantic-segmentation/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/train"
    val_dir = "/kaggle/input/datasets/divyanshsharma23/semantic-segmentation/Offroad_Segmentation_Training_Dataset/Offroad_Segmentation_Training_Dataset/val"

    # Create datasets
    
    trainset = MaskDataset(data_dir=data_dir, transform=transform_train, mask_transform=mask_transform)
    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=False)

    valset = MaskDataset(data_dir=val_dir, transform=transform, mask_transform=mask_transform)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Training samples: {len(trainset)}")
    print(f"Validation samples: {len(valset)}")

    # Load DINOv2 backbone
    print("Loading DINOv2 backbone...")
    BACKBONE_SIZE = "small"
    backbone_archs = {
        "small": "vits14",
        "base": "vitb14_reg",
        "large": "vitl14_reg",
        "giant": "vitg14_reg",
    }
    backbone_arch = backbone_archs[BACKBONE_SIZE]
    backbone_name = f"dinov2_{backbone_arch}"

    backbone_model = torch.hub.load(repo_or_dir="facebookresearch/dinov2", model=backbone_name,trust_repo=True)
    backbone_model.eval()
    backbone_model.to(device)
    print("Backbone loaded successfully!")
    # ADD ALL THIS RIGHT AFTER ↓
    backbone_model.eval()
    for param in backbone_model.parameters():
       param.requires_grad = False

    # for param in backbone_model.blocks[-2:].parameters():
    #    param.requires_grad = True // FROZEN  METRICES WOULD TAKE MORE TIME 

    # Get embedding dimension from backbone
    imgs, _ = next(iter(train_loader))
    imgs = imgs.to(device)
    with torch.no_grad():
        output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]
    n_embedding = output.shape[2]
    print(f"Embedding dimension: {n_embedding}")
    print(f"Patch tokens shape: {output.shape}")

    # Create segmentation head
    classifier = ASPPFPNSegmentationHead(
        in_channels=n_embedding,
        out_channels=n_classes,
        
        tokenW=w // 14,
        tokenH=h // 14
    )
    classifier = classifier.to(device)

    # Loss and optimizer
    loss_fct = CombinedLoss(num_classes=n_classes)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=n_epochs,
    pct_start=0.05  # 10% warmup
)

    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_iou': [],
        'val_iou': [],
        'train_dice': [],
        'val_dice': [],
        'train_pixel_acc': [],
        'val_pixel_acc': []
    }

    # Training loop
    print("\nStarting training...")
    print("=" * 80)
    best_val_iou = 0.0  
    epoch_pbar = tqdm(range(n_epochs), desc="Training", unit="epoch")
    for epoch in epoch_pbar:
        # Training phase
        classifier.train()
        train_losses = []
        

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Train]", 
                          leave=False, unit="batch")
        for imgs, labels in train_pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            
            with torch.no_grad():
                   output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]

            logits = classifier(output.to(device))
            outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)  

            labels = labels.squeeze(dim=1).long()
            labels = F.interpolate(labels.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest").squeeze(1).long()

            loss = loss_fct(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()
            train_losses.append(loss.item())
            train_pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Validation phase
        classifier.eval()
        val_losses = []

        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{n_epochs} [Val]", 
                        leave=False, unit="batch")
        with torch.no_grad():
            for imgs, labels in val_pbar:
                imgs, labels = imgs.to(device), labels.to(device)

                output = backbone_model.forward_features(imgs)["x_norm_patchtokens"]
                logits = classifier(output.to(device))
                outputs = F.interpolate(logits, size=imgs.shape[2:], mode="bilinear", align_corners=False)  # MISSING

                labels = labels.squeeze(dim=1).long()
                labels = F.interpolate(labels.unsqueeze(1).float(), size=outputs.shape[2:], mode="nearest").squeeze(1).long()
                loss = loss_fct(outputs, labels)
                val_losses.append(loss.item())
                val_pbar.set_postfix(loss=f"{loss.item():.4f}")

        # Calculate metrics
        train_iou, train_dice, train_pixel_acc = evaluate_metrics(
            classifier, backbone_model, train_loader, device, num_classes=n_classes
        )
        val_iou, val_dice, val_pixel_acc = evaluate_metrics(
            classifier, backbone_model, val_loader, device, num_classes=n_classes
           )

        # Store history
        epoch_train_loss = np.mean(train_losses)
        epoch_val_loss = np.mean(val_losses)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_pixel_acc'].append(train_pixel_acc)
        history['val_pixel_acc'].append(val_pixel_acc)
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save(classifier.state_dict(), os.path.join(script_dir, "best_segmentation_head.pth"))
            print(f"  Saved best model with Val IoU: {best_val_iou:.4f}")
        
        # / added a learning schedular
        # Update epoch progress bar with metrics
        epoch_pbar.set_postfix(
            train_loss=f"{epoch_train_loss:.3f}",
            val_loss=f"{epoch_val_loss:.3f}",
            val_iou=f"{val_iou:.3f}",
            val_acc=f"{val_pixel_acc:.3f}"
        )

    # Save plots
    print("\nSaving training curves...")
    save_training_plots(history, output_dir)
    save_history_to_file(history, output_dir)

        
    # Save model (in scripts directory)
    model_path = os.path.join(script_dir, "segmentation_head.pth")
    torch.save(classifier.state_dict(), model_path)
    print(f"Saved model to '{model_path}'")

    # Final evaluation
    print("\nFinal evaluation results:")
    print(f"  Final Val Loss:     {history['val_loss'][-1]:.4f}")
    print(f"  Final Val IoU:      {history['val_iou'][-1]:.4f}")
    print(f"  Final Val Dice:     {history['val_dice'][-1]:.4f}")
    print(f"  Final Val Accuracy: {history['val_pixel_acc'][-1]:.4f}")

    print("\nTraining complete!")

In [10]:
if __name__ == "__main__":
    main()


Using device: cuda
Training samples: 2857
Validation samples: 317
Loading DINOv2 backbone...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /kaggle/working/torch_hub/main.zip


/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/torch_hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /kaggle/working/torch_hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 240MB/s]


Backbone loaded successfully!
Embedding dimension: 384
Patch tokens shape: torch.Size([4, 646, 384])

Starting training...


...

  Saved best model with Val IoU: 0.5199



Epoch 9/10 [Train]: 100%|█████████▉| 714/715 [01:27<00:00,  8.23batch/s, loss=0.4907]
                                                                                     
Epoch 9/10 [Val]:  99%|█████████▉| 79/80 [00:08<00:00,  9.91batch/s, loss=0.4780]
                                                                                 
Evaluating: 100%|█████████▉| 714/715 [01:16<00:00,  9.96batch/s]
                                                                
Training:  90%|█████████ | 9/10 [28:41<03:03, 183.24s/epoch, train_loss=0.423, val_acc=0.835, val_iou=0.523, val_loss=0.424]

  Saved best model with Val IoU: 0.5226



Epoch 10/10 [Train]: 100%|█████████▉| 714/715 [01:28<00:00,  8.21batch/s, loss=0.5801]
                                                                                      
Epoch 10/10 [Val]: 100%|██████████| 80/80 [00:08<00:00, 11.71batch/s, loss=0.4750]
                                                                                  
Evaluating: 100%|█████████▉| 714/715 [01:16<00:00,  9.77batch/s]
                                                                
Training: 100%|██████████| 10/10 [31:43<00:00, 190.36s/epoch, train_loss=0.425, val_acc=0.836, val_iou=0.522, val_loss=0.424]



Saving training curves...
Saved training curves to '/kaggle/working/train_stats/training_curves.png'
Saved IoU curves to '/kaggle/working/train_stats/iou_curves.png'
Saved Dice curves to '/kaggle/working/train_stats/dice_curves.png'
Saved combined metrics curves to '/kaggle/working/train_stats/all_metrics_curves.png'
Saved evaluation metrics to /kaggle/working/train_stats/evaluation_metrics.txt
Saved model to '/kaggle/working/segmentation_head.pth'

Final evaluation results:
  Final Val Loss:     0.4242
  Final Val IoU:      0.5222
  Final Val Dice:     0.6609
  Final Val Accuracy: 0.8355

Training complete!
